In [1]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.9 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from faker import Faker

fake = Faker(["pl_PL", "en_GB", "de_DE", "es_ES"])
Faker.seed(42)

for i in range(5):
    print(fake.name(), "|", fake.date_of_birth(minimum_age=18, maximum_age=90), "|", fake.current_country())

Jacek Dreszer | 1946-01-28 | Poland
Filip Glapiak | 1951-11-07 | Poland
Victor Harrison-Burton | 1978-11-28 | Poland
Phillip Cook-Bryan | 1943-05-09 | Spain
pani Rozalia Dykas | 1989-02-28 | Poland


In [6]:
import pandas as pd

customers = []
for i in range(5000):
    customers.append({
        "customer_id": f"C{i:05d}",
        "name": fake.first_name() + " " + fake.last_name(),
        "dob": fake.date_of_birth(minimum_age=18, maximum_age=90),
        "nationality": fake.current_country(),
    })

customers = pd.DataFrame(customers)
customers.head(10)

,customer_id,name,dob,nationality
0,C00000,Justin Patel,1959-01-07,Spain
1,C00001,Natalie Danel,1990-12-19,Germany
2,C00002,Wiktor Cordero,1965-10-05,Spain
3,C00003,Norma Valderrama,1971-06-08,Spain
4,C00004,Donato Davis,1995-07-19,United Kingdom
5,C00005,Eleuterio Kotlarek,1987-03-24,United Kingdom
6,C00006,Marian Loos,1961-09-19,Germany
7,C00007,Joanne Alfaro,1990-08-04,Spain
8,C00008,Alan Łosiewicz,1995-02-26,Germany
9,C00009,Juliusz Gardiner,1942-05-03,United Kingdom


In [7]:
customers["label"] = "clean"

people = pd.read_csv("../data/processed/people.csv")
known = people.dropna(subset=["dob", "nationality"])

true_hits = known.sample(20, random_state=1)
lookalikes = people.sample(200, random_state=2)

planted = []
for i, (_, p) in enumerate(true_hits.iterrows()):
    planted.append({"customer_id": f"T{i:05d}", "name": p["name"],
                    "dob": p["dob"], "nationality": p["nationality"],
                    "label": "true_match"})
for i, (_, p) in enumerate(lookalikes.iterrows()):
    planted.append({"customer_id": f"L{i:05d}", "name": p["name"],
                    "dob": fake.date_of_birth(minimum_age=18, maximum_age=90),
                    "nationality": fake.current_country(),
                    "label": "lookalike"})

customers = pd.concat([customers, pd.DataFrame(planted)]).reset_index(drop=True)
customers["label"].value_counts()

label
clean         5000
lookalike      200
true_match      20
Name: count, dtype: int64

In [8]:
customers = customers.sample(frac=1, random_state=3).reset_index(drop=True)
customers.to_csv("../data/processed/customers.csv", index=False)
customers.head(10)

,customer_id,name,dob,nationality,label
0,C00281,Denise Borrego,1975-12-11,Germany,clean
1,C00500,Gunter Vicente,1959-10-10,Spain,clean
2,C04253,Alexei Sauer,1954-07-11,Poland,clean
3,C03677,Alan Żelazo,1981-12-28,United Kingdom,clean
4,C02454,Holm Todd,1949-08-03,Spain,clean
5,L00105,"VAKHROMEYEV, Ivan Vasilyevich",1960-09-06,Poland,lookalike
6,C01185,Itziar Burgess,2003-06-05,Poland,clean
7,C01456,Liam Jimenez,1986-01-30,Poland,clean
8,C01681,Reinhilde Romejko,1964-04-10,Spain,clean
9,C00934,Apolonia Howe,2003-05-08,Poland,clean
